# Document Processing, Similarity, and Clustering

Advanced document processing pipeline including similarity matching, clustering, document classification, and information retrieval.

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict, Optional, Tuple, Union, Any
from dataclasses import dataclass
import json
from pathlib import Path
from collections import defaultdict, Counter
import warnings
import time

# Document processing
import PyPDF2
import docx
from bs4 import BeautifulSoup
import markdown
import re

# NLP libraries
import spacy
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Doc2Vec, Word2Vec
from gensim.models.doc2vec import TaggedDocument

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, MeanShift
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import StandardScaler

# Deep learning
import torch

try:
    from sentence_transformers import SentenceTransformer

    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

warnings.filterwarnings("ignore")

# Set style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Document Processing Pipeline initialized!")

## 1. Document Loading and Preprocessing

In [ ]:
class DocumentProcessor:
    """Process various document formats."""

    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.supported_formats = [
            ".txt",
            ".pdf",
            ".docx",
            ".html",
            ".md",
            ".json",
            ".csv",
        ]

    def load_document(self, file_path: str) -> str:
        """Load document from various formats."""

        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        extension = path.suffix.lower()

        if extension == ".txt":
            return self._load_text(path)
        elif extension == ".pdf":
            return self._load_pdf(path)
        elif extension == ".docx":
            return self._load_docx(path)
        elif extension == ".html":
            return self._load_html(path)
        elif extension == ".md":
            return self._load_markdown(path)
        elif extension == ".json":
            return self._load_json(path)
        elif extension == ".csv":
            return self._load_csv(path)
        else:
            raise ValueError(f"Unsupported format: {extension}")

    def _load_text(self, path: Path) -> str:
        """Load plain text file."""
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    def _load_pdf(self, path: Path) -> str:
        """Load PDF file."""
        text = ""
        with open(path, "rb") as f:
            pdf_reader = PyPDF2.PdfReader(f)
            for page in pdf_reader.pages:
                text += page.extract_text()
        return text

    def _load_docx(self, path: Path) -> str:
        """Load Word document."""
        doc = docx.Document(path)
        return "\n".join([para.text for para in doc.paragraphs])

    def _load_html(self, path: Path) -> str:
        """Load HTML file and extract text."""
        with open(path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
            return soup.get_text()

    def _load_markdown(self, path: Path) -> str:
        """Load Markdown file and convert to text."""
        with open(path, "r", encoding="utf-8") as f:
            md_text = f.read()
            html = markdown.markdown(md_text)
            soup = BeautifulSoup(html, "html.parser")
            return soup.get_text()

    def _load_json(self, path: Path) -> str:
        """Load JSON file."""
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return json.dumps(data, indent=2)

    def _load_csv(self, path: Path) -> str:
        """Load CSV file."""
        df = pd.read_csv(path)
        return df.to_string()

    def preprocess_document(
        self,
        text: str,
        remove_stopwords: bool = True,
        lemmatize: bool = True,
        remove_punctuation: bool = True,
        lowercase: bool = True,
    ) -> str:
        """Preprocess document text."""

        # Process with spaCy
        doc = self.nlp(text)

        tokens = []
        for token in doc:
            # Skip punctuation
            if remove_punctuation and token.is_punct:
                continue

            # Skip stopwords
            if remove_stopwords and token.is_stop:
                continue

            # Get token text
            if lemmatize:
                token_text = token.lemma_
            else:
                token_text = token.text

            if lowercase:
                token_text = token_text.lower()

            tokens.append(token_text)

        return " ".join(tokens)

    def extract_metadata(self, file_path: str) -> Dict:
        """Extract metadata from document."""

        path = Path(file_path)
        text = self.load_document(file_path)
        doc = self.nlp(text[:1000000])  # Limit for performance

        metadata = {
            "file_name": path.name,
            "file_size": path.stat().st_size,
            "extension": path.suffix,
            "word_count": len([token for token in doc if not token.is_punct]),
            "sentence_count": len(list(doc.sents)),
            "entities": [(ent.text, ent.label_) for ent in doc.ents][:10],
            "language": doc.lang_,
        }

        return metadata

## 2. Document Similarity and Matching

In [ ]:
class DocumentSimilarity:
    """Calculate document similarity using various methods."""

    def __init__(self, method: str = "tfidf"):
        self.method = method
        self.vectorizer = None
        self.doc_vectors = None
        self.documents = None

        if method == "sentence_bert" and SENTENCE_TRANSFORMERS_AVAILABLE:
            self.encoder = SentenceTransformer("all-MiniLM-L6-v2")

    def fit(self, documents: List[str]):
        """Fit similarity model on documents."""

        self.documents = documents

        if self.method == "tfidf":
            self._fit_tfidf(documents)
        elif self.method == "doc2vec":
            self._fit_doc2vec(documents)
        elif self.method == "sentence_bert":
            self._fit_sentence_bert(documents)
        elif self.method == "word2vec_avg":
            self._fit_word2vec_avg(documents)
        else:
            raise ValueError(f"Unknown method: {self.method}")

    def _fit_tfidf(self, documents: List[str]):
        """Fit TF-IDF vectorizer."""
        self.vectorizer = TfidfVectorizer(
            max_features=5000, ngram_range=(1, 3), min_df=2, max_df=0.95
        )
        self.doc_vectors = self.vectorizer.fit_transform(documents)

    def _fit_doc2vec(self, documents: List[str]):
        """Fit Doc2Vec model."""
        # Prepare tagged documents
        tagged_docs = [
            TaggedDocument(doc.split(), [i]) for i, doc in enumerate(documents)
        ]

        # Train Doc2Vec
        self.doc2vec_model = Doc2Vec(
            vector_size=100, window=5, min_count=2, workers=4, epochs=40
        )

        self.doc2vec_model.build_vocab(tagged_docs)
        self.doc2vec_model.train(
            tagged_docs,
            total_examples=self.doc2vec_model.corpus_count,
            epochs=self.doc2vec_model.epochs,
        )

        # Get document vectors
        self.doc_vectors = np.array(
            [self.doc2vec_model.dv[i] for i in range(len(documents))]
        )

    def _fit_sentence_bert(self, documents: List[str]):
        """Fit Sentence-BERT embeddings."""
        if not SENTENCE_TRANSFORMERS_AVAILABLE:
            raise ImportError("sentence-transformers not available")

        self.doc_vectors = self.encoder.encode(
            documents, convert_to_tensor=False, show_progress_bar=True
        )

    def _fit_word2vec_avg(self, documents: List[str]):
        """Fit Word2Vec with average pooling."""
        # Tokenize documents
        tokenized_docs = [doc.split() for doc in documents]

        # Train Word2Vec
        self.word2vec_model = Word2Vec(
            tokenized_docs, vector_size=100, window=5, min_count=2, workers=4
        )

        # Get document vectors by averaging word vectors
        doc_vectors = []
        for tokens in tokenized_docs:
            word_vecs = []
            for token in tokens:
                if token in self.word2vec_model.wv:
                    word_vecs.append(self.word2vec_model.wv[token])

            if word_vecs:
                doc_vectors.append(np.mean(word_vecs, axis=0))
            else:
                doc_vectors.append(np.zeros(100))

        self.doc_vectors = np.array(doc_vectors)

    def calculate_similarity(self, doc1_idx: int, doc2_idx: int) -> float:
        """Calculate similarity between two documents."""

        if self.doc_vectors is None:
            raise ValueError("Model not fitted yet")

        if hasattr(self.doc_vectors, "toarray"):
            # Sparse matrix
            vec1 = self.doc_vectors[doc1_idx].toarray().flatten()
            vec2 = self.doc_vectors[doc2_idx].toarray().flatten()
        else:
            vec1 = self.doc_vectors[doc1_idx]
            vec2 = self.doc_vectors[doc2_idx]

        # Cosine similarity
        similarity = cosine_similarity([vec1], [vec2])[0][0]

        return similarity

    def find_similar(
        self, query: Union[str, int], top_k: int = 5
    ) -> List[Tuple[int, float]]:
        """Find similar documents to query."""

        if isinstance(query, str):
            # Query is new text
            query_vector = self._vectorize_query(query)
        else:
            # Query is document index
            if hasattr(self.doc_vectors, "toarray"):
                query_vector = self.doc_vectors[query].toarray().flatten()
            else:
                query_vector = self.doc_vectors[query]

        # Calculate similarities
        if hasattr(self.doc_vectors, "toarray"):
            similarities = cosine_similarity(
                [query_vector], self.doc_vectors.toarray()
            )[0]
        else:
            similarities = cosine_similarity([query_vector], self.doc_vectors)[0]

        # Get top-k
        top_indices = np.argsort(similarities)[-top_k - 1 : -1][::-1]

        skip_idx = query if isinstance(query, int) else None
        results = [
            (idx, similarities[idx])
            for idx in top_indices
            if skip_idx is None or idx != skip_idx
        ]

        return results[:top_k]

    def _vectorize_query(self, query: str):
        """Vectorize query text."""

        if self.method == "tfidf":
            return self.vectorizer.transform([query]).toarray().flatten()
        elif self.method == "doc2vec":
            return self.doc2vec_model.infer_vector(query.split())
        elif self.method == "sentence_bert":
            return self.encoder.encode([query])[0]
        elif self.method == "word2vec_avg":
            tokens = query.split()
            word_vecs = []
            for token in tokens:
                if token in self.word2vec_model.wv:
                    word_vecs.append(self.word2vec_model.wv[token])

            if word_vecs:
                return np.mean(word_vecs, axis=0)
            else:
                return np.zeros(100)

    def get_similarity_matrix(self) -> np.ndarray:
        """Get full similarity matrix."""

        if self.doc_vectors is None:
            raise ValueError("Model not fitted yet")

        if hasattr(self.doc_vectors, "toarray"):
            return cosine_similarity(self.doc_vectors.toarray())
        else:
            return cosine_similarity(self.doc_vectors)

## 3. Document Clustering

In [ ]:
class DocumentClustering:
    """Document clustering with multiple algorithms."""

    def __init__(self, n_clusters: int = 5, method: str = "kmeans"):
        self.n_clusters = n_clusters
        self.method = method
        self.clusterer = None
        self.labels = None

    def fit_predict(self, doc_vectors: np.ndarray) -> np.ndarray:
        """Fit clustering model and predict labels."""

        # Convert sparse matrix if needed
        if hasattr(doc_vectors, "toarray"):
            doc_vectors = doc_vectors.toarray()

        if self.method == "kmeans":
            self.clusterer = KMeans(
                n_clusters=self.n_clusters, random_state=42, n_init=10
            )
        elif self.method == "dbscan":
            self.clusterer = DBSCAN(eps=0.3, min_samples=2, metric="cosine")
        elif self.method == "hierarchical":
            self.clusterer = AgglomerativeClustering(
                n_clusters=self.n_clusters, linkage="ward"
            )
        elif self.method == "meanshift":
            self.clusterer = MeanShift(bandwidth=None, bin_seeding=True)
        else:
            raise ValueError(f"Unknown method: {self.method}")

        self.labels = self.clusterer.fit_predict(doc_vectors)

        # Update n_clusters for methods that determine it automatically
        if self.method in ["dbscan", "meanshift"]:
            self.n_clusters = len(set(self.labels)) - (1 if -1 in self.labels else 0)

        return self.labels

    def evaluate(self, doc_vectors: np.ndarray) -> Dict:
        """Evaluate clustering quality."""

        if self.labels is None:
            raise ValueError("Model not fitted yet")

        # Convert sparse matrix if needed
        if hasattr(doc_vectors, "toarray"):
            doc_vectors = doc_vectors.toarray()

        metrics = {}

        # Only calculate if we have meaningful clusters
        unique_labels = set(self.labels)
        if len(unique_labels) > 1 and len(unique_labels) < len(self.labels):
            # Filter out noise points for metrics
            if -1 in self.labels:
                mask = self.labels != -1
                if mask.sum() > 1:
                    metrics["silhouette"] = silhouette_score(
                        doc_vectors[mask], self.labels[mask]
                    )
                    metrics["davies_bouldin"] = davies_bouldin_score(
                        doc_vectors[mask], self.labels[mask]
                    )
                    metrics["calinski_harabasz"] = calinski_harabasz_score(
                        doc_vectors[mask], self.labels[mask]
                    )
            else:
                metrics["silhouette"] = silhouette_score(doc_vectors, self.labels)
                metrics["davies_bouldin"] = davies_bouldin_score(
                    doc_vectors, self.labels
                )
                metrics["calinski_harabasz"] = calinski_harabasz_score(
                    doc_vectors, self.labels
                )

        # Cluster statistics
        label_counts = Counter(self.labels)
        metrics["n_clusters"] = self.n_clusters
        metrics["cluster_sizes"] = dict(label_counts)
        metrics["noise_points"] = label_counts.get(-1, 0)

        return metrics

    def get_cluster_topics(
        self,
        documents: List[str],
        vectorizer: TfidfVectorizer = None,
        n_terms: int = 10,
    ) -> Dict:
        """Extract top terms for each cluster."""

        if self.labels is None:
            raise ValueError("Model not fitted yet")

        # Fit vectorizer if not provided
        if vectorizer is None:
            vectorizer = TfidfVectorizer(
                max_features=1000, ngram_range=(1, 2), stop_words="english"
            )
            vectorizer.fit(documents)

        feature_names = vectorizer.get_feature_names_out()

        cluster_topics = {}

        for cluster_id in range(self.n_clusters):
            # Get documents in cluster
            cluster_docs = [
                doc for i, doc in enumerate(documents) if self.labels[i] == cluster_id
            ]

            if cluster_docs:
                # Get TF-IDF scores for cluster
                cluster_tfidf = vectorizer.transform(cluster_docs)

                # Average TF-IDF scores
                avg_scores = np.asarray(cluster_tfidf.mean(axis=0)).flatten()

                # Get top terms
                top_indices = avg_scores.argsort()[-n_terms:][::-1]
                top_terms = [(feature_names[i], avg_scores[i]) for i in top_indices]

                cluster_topics[cluster_id] = {
                    "size": len(cluster_docs),
                    "top_terms": top_terms,
                }

        return cluster_topics

    def visualize_clusters(
        self, doc_vectors: np.ndarray, documents: List[str] = None, method: str = "tsne"
    ) -> None:
        """Visualize document clusters."""

        if self.labels is None:
            raise ValueError("Model not fitted yet")

        # Convert sparse matrix if needed
        if hasattr(doc_vectors, "toarray"):
            doc_vectors = doc_vectors.toarray()

        # Dimensionality reduction
        if method == "tsne":
            reducer = TSNE(n_components=2, random_state=42)
        elif method == "pca":
            reducer = PCA(n_components=2, random_state=42)
        else:
            raise ValueError(f"Unknown visualization method: {method}")

        coords = reducer.fit_transform(doc_vectors)

        # Create plot
        plt.figure(figsize=(12, 8))

        # Plot points
        unique_labels = set(self.labels)
        colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))

        for label, color in zip(unique_labels, colors):
            mask = self.labels == label
            if label == -1:
                # Noise points
                plt.scatter(
                    coords[mask, 0],
                    coords[mask, 1],
                    c="gray",
                    marker="x",
                    s=30,
                    alpha=0.5,
                    label="Noise",
                )
            else:
                plt.scatter(
                    coords[mask, 0],
                    coords[mask, 1],
                    c=[color],
                    s=50,
                    alpha=0.7,
                    label=f"Cluster {label}",
                )

        plt.title(f"Document Clustering Visualization ({method.upper()})")
        plt.xlabel(f"{method.upper()} 1")
        plt.ylabel(f"{method.upper()} 2")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

## 4. Document Network Analysis

In [ ]:
class DocumentNetwork:
    """Build and analyze document networks."""

    def __init__(self, similarity_threshold: float = 0.5):
        self.similarity_threshold = similarity_threshold
        self.graph = None
        self.similarity_matrix = None

    def build_network(
        self, similarity_matrix: np.ndarray, document_names: List[str] = None
    ) -> nx.Graph:
        """Build document network from similarity matrix."""

        self.similarity_matrix = similarity_matrix
        n_docs = similarity_matrix.shape[0]

        if document_names is None:
            document_names = [f"Doc_{i}" for i in range(n_docs)]

        # Create graph
        self.graph = nx.Graph()

        # Add nodes
        for i, name in enumerate(document_names):
            self.graph.add_node(i, name=name)

        # Add edges based on similarity threshold
        for i in range(n_docs):
            for j in range(i + 1, n_docs):
                similarity = similarity_matrix[i, j]
                if similarity >= self.similarity_threshold:
                    self.graph.add_edge(i, j, weight=similarity)

        return self.graph

    def analyze_network(self) -> Dict:
        """Analyze network properties."""

        if self.graph is None:
            raise ValueError("Network not built yet")

        analysis = {}

        # Basic statistics
        analysis["n_nodes"] = self.graph.number_of_nodes()
        analysis["n_edges"] = self.graph.number_of_edges()
        analysis["density"] = nx.density(self.graph)

        # Centrality measures
        analysis["degree_centrality"] = nx.degree_centrality(self.graph)
        analysis["betweenness_centrality"] = nx.betweenness_centrality(self.graph)
        analysis["closeness_centrality"] = nx.closeness_centrality(self.graph)
        analysis["eigenvector_centrality"] = nx.eigenvector_centrality_numpy(self.graph)

        # Community detection
        communities = nx.community.greedy_modularity_communities(self.graph)
        analysis["n_communities"] = len(communities)
        analysis["communities"] = [list(community) for community in communities]
        analysis["modularity"] = nx.community.modularity(self.graph, communities)

        # Connected components
        analysis["n_components"] = nx.number_connected_components(self.graph)
        analysis["largest_component_size"] = len(
            max(nx.connected_components(self.graph), key=len)
        )

        # Clustering coefficient
        analysis["avg_clustering"] = nx.average_clustering(self.graph)

        return analysis

    def find_key_documents(self, top_k: int = 10) -> List[Tuple[int, float]]:
        """Find most central/important documents."""

        if self.graph is None:
            raise ValueError("Network not built yet")

        # Use PageRank as importance measure
        pagerank = nx.pagerank(self.graph)

        # Sort by PageRank score
        sorted_docs = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)

        return sorted_docs[:top_k]

    def visualize_network(
        self, node_labels: Dict = None, layout: str = "spring"
    ) -> None:
        """Visualize document network."""

        if self.graph is None:
            raise ValueError("Network not built yet")

        plt.figure(figsize=(14, 10))

        # Choose layout
        if layout == "spring":
            pos = nx.spring_layout(self.graph, k=0.5, iterations=50)
        elif layout == "circular":
            pos = nx.circular_layout(self.graph)
        elif layout == "kamada_kawai":
            pos = nx.kamada_kawai_layout(self.graph)
        else:
            pos = nx.random_layout(self.graph)

        # Node sizes based on degree
        node_sizes = [300 * self.graph.degree(n) for n in self.graph.nodes()]

        # Draw nodes
        nx.draw_networkx_nodes(
            self.graph, pos, node_size=node_sizes, node_color="lightblue", alpha=0.7
        )

        # Draw edges
        edges = self.graph.edges()
        weights = [self.graph[u][v]["weight"] for u, v in edges]

        nx.draw_networkx_edges(
            self.graph, pos, width=[w * 3 for w in weights], alpha=0.3
        )

        # Draw labels
        if node_labels is None:
            node_labels = {
                n: self.graph.nodes[n].get("name", str(n)) for n in self.graph.nodes()
            }

        nx.draw_networkx_labels(self.graph, pos, labels=node_labels, font_size=8)

        plt.title("Document Network Visualization")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 5. Example Usage and Demonstrations

In [ ]:
def demo_document_processing():
    """Demonstrate document processing capabilities."""

    print("\n" + "=" * 50)
    print("Document Processing and Analysis Demo")
    print("=" * 50)

    # Sample documents
    documents = [
        "Machine learning is a subset of artificial intelligence that enables computers to learn from data.",
        "Deep learning uses neural networks with multiple layers to process complex patterns.",
        "Natural language processing helps computers understand and generate human language.",
        "Computer vision allows machines to interpret and analyze visual information from images.",
        "Reinforcement learning trains agents through trial and error using rewards and penalties.",
        "Data science combines statistics, programming, and domain expertise to extract insights.",
        "Big data analytics processes large volumes of structured and unstructured data.",
        "Cloud computing provides on-demand access to computing resources over the internet.",
        "Cybersecurity protects systems, networks, and data from digital attacks.",
        "Blockchain is a distributed ledger technology for secure transactions.",
    ]

    # 1. Document Similarity
    print("\n1. Document Similarity Analysis")
    similarity = DocumentSimilarity(method="tfidf")
    similarity.fit(documents)

    # Find similar documents
    query = "What is artificial intelligence and machine learning?"
    similar_docs = similarity.find_similar(query, top_k=3)

    print(f"\nQuery: {query}")
    print("Similar documents:")
    for idx, score in similar_docs:
        print(f"  - Doc {idx}: {documents[idx][:50]}... (Score: {score:.3f})")

    # 2. Document Clustering
    print("\n2. Document Clustering")
    clustering = DocumentClustering(n_clusters=3, method="kmeans")
    labels = clustering.fit_predict(similarity.doc_vectors)

    # Get cluster topics
    cluster_topics = clustering.get_cluster_topics(documents)

    print("\nCluster Topics:")
    for cluster_id, info in cluster_topics.items():
        top_terms = ", ".join([term for term, _ in info["top_terms"][:5]])
        print(f"  Cluster {cluster_id} ({info['size']} docs): {top_terms}")

    # Evaluate clustering
    metrics = clustering.evaluate(similarity.doc_vectors)
    print(f"\nClustering Metrics:")
    if "silhouette" in metrics:
        print(f"  Silhouette Score: {metrics['silhouette']:.3f}")

    # 3. Document Network
    print("\n3. Document Network Analysis")
    network = DocumentNetwork(similarity_threshold=0.2)

    # Build network
    sim_matrix = similarity.get_similarity_matrix()
    graph = network.build_network(
        sim_matrix, [f"Doc{i}" for i in range(len(documents))]
    )

    # Analyze network
    analysis = network.analyze_network()
    print(f"\nNetwork Statistics:")
    print(f"  Nodes: {analysis['n_nodes']}")
    print(f"  Edges: {analysis['n_edges']}")
    print(f"  Density: {analysis['density']:.3f}")
    print(f"  Communities: {analysis['n_communities']}")
    print(f"  Modularity: {analysis['modularity']:.3f}")

    # Find key documents
    key_docs = network.find_key_documents(top_k=3)
    print("\nKey Documents (by PageRank):")
    for doc_id, score in key_docs:
        print(f"  - Doc {doc_id}: {documents[doc_id][:50]}... (Score: {score:.3f})")

    return similarity, clustering, network


# Run demonstration
if __name__ == "__main__":
    results = demo_document_processing()
    print("\nDocument processing demonstration completed!")

## Summary

This document processing notebook provides:

1. **Document Loading**: Support for PDF, Word, HTML, Markdown, JSON, CSV formats
2. **Document Similarity**: TF-IDF, Doc2Vec, Sentence-BERT, Word2Vec averaging
3. **Document Clustering**: K-Means, DBSCAN, Hierarchical, MeanShift clustering
4. **Network Analysis**: Document networks with community detection and centrality analysis
5. **Visualization**: Cluster visualization, network graphs, similarity heatmaps
6. **Evaluation**: Comprehensive metrics for clustering and similarity

The pipeline is modular and can be easily extended for specific document processing tasks.